# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library, referencing all entities (record sets, fields, columns) by their `@id` per best practices.

### Dataset Source
The dataset Croissant schema is available at:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure mlcroissant is installed and up-to-date
!pip install --quiet --upgrade mlcroissant

## 1. Data Loading
We use `mlcroissant` to load both the metadata and the records from the dataset.
All interaction with the dataset is through its Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Pretty print info about the dataset
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n\nAuthors:")
for author in getattr(metadata, 'author', []):
    if hasattr(author, 'name'):
        print(f" - {author.name}")
print("\nLicense:", getattr(metadata, 'license', ''))
print("Keywords:", getattr(metadata, 'keywords', ''))

## 2. Data Overview

Let's inspect the available **record sets** and their IDs, along with their fields. We will reference all by their `@id`, which is best practice in Croissant.

_Note: In Croissant, "record sets" correspond to major tables/files/collections of related data._

In [ ]:
# Display info about all record sets using their @id
record_sets = getattr(metadata, 'record_set', [])
if not record_sets:
    print("No record sets found in the root metadata. Will look in distribution metadata as per Croissant specification...")
    # Sometimes recordSets are defined under distributions in Croissant datasets
    record_sets = []
    for dist in getattr(metadata, 'distribution', []):
        if hasattr(dist, 'record_set'):
            rs = dist.record_set
            if isinstance(rs, list):
                record_sets.extend(rs)
            elif rs is not None:
                record_sets.append(rs)
    if not record_sets:
        print("No record sets found under distribution. Please check the schema or download and inspect manually.")

if record_sets:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        rs_id = getattr(rs, '@id', '<missing @id>')
        rs_name = getattr(rs, 'name', '')
        print(f"- RecordSet @id: {rs_id}\n  Name: {rs_name}")
        fields = getattr(rs, 'field', [])
        if isinstance(fields, dict) or hasattr(fields, '@id'):
            fields = [fields] # handle singleton field
        print(f"  Fields:")
        for fld in fields:
            f_id = getattr(fld, '@id', '<missing @id>')
            f_name = getattr(fld, 'name', '')
            print(f"    * @id: {f_id}  Name: {f_name}")
        print()
else:
    print("No record sets found. Please check the Croissant specification for this dataset.")

## 3. Data Extraction

Let's extract the data from available record set(s), using their `@id`. We'll load the record set(s) as pandas DataFrame(s) for further processing. All field references are by `@id`.

**Note:**
If the dataset has multiple record sets, we'll extract them all and display their columns for inspection.

In [ ]:
# List record set @ids
record_set_ids = []
record_set_names = {}
if record_sets:
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        if rs_id:
            record_set_ids.append(rs_id)
            record_set_names[rs_id] = getattr(rs, 'name', '')
        
print("Record set @ids:")
for i, rsid in enumerate(record_set_ids):
    print(f" {i+1}: {rsid} -- {record_set_names[rsid]}")

# Load all record sets as DataFrames using their @id
dfs = {}
for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dfs[rsid] = df
        print(f"\nLoaded record set '{rsid}' with {len(df)} records and columns:")
        print(df.columns.tolist())
    except Exception as e:
        print(f"Failed to load records for {rsid}: {e}")

# For demonstration, select the first available record set (by @id):
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"\nFirst record set selected: {example_record_set_id}")
    display(dfs[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We demonstrate EDA on the first available record set, referencing all columns (fields) by their `@id` as required. _Modify the variable `numeric_field_id` below to use a desired numeric field's `@id` as found in the previous overview code outputs._

In [ ]:
# Example: Choose a numeric field/column for demonstration purposes
# You should update numeric_field_id to a real field @id as discovered above.
example_rs_id = example_record_set_id  # Use the first record set found
df = dfs[example_rs_id]
print(f"Data columns in record set '{example_rs_id}':", df.columns.tolist())

# You must set this! Based on the output above, pick a numeric column @id or name.
numeric_field_id = df.columns[0]  # e.g., '@id:cr:field:MW' or the actual numeric column id

# Filter for values above a threshold, normalize, and display grouped mean by another field
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean()  # Use mean as an example threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()

    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field if available (e.g. by a categorical column)
    group_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]
    if group_candidates:
        group_field_id = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (showing mean {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print("No categorical field found to group by.")
else:
    print(f"Selected field '{numeric_field_id}' is not numeric. Please choose a numeric field from: {df.columns.tolist()}")

## 5. Visualization

Let's plot the distribution of the selected numeric field for the record set reference by its `@id`. All plotting and documentation will reference fields by `@id` for CROISSANT compliance.

In [ ]:
import matplotlib.pyplot as plt

if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8, 5))
    df[numeric_field_id].hist(bins=30, alpha=0.7)
    plt.title(f"Distribution of field '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
else:
    print(f"Field '{numeric_field_id}' is not numeric and can't be visualized as a histogram.")

## 6. Conclusion

In this notebook, we:
- Loaded the FAIR² Croissant dataset and programmatically explored all record sets, fields, and columns by their `@id`.
- Extracted and inspected data dynamically for each record set.
- Demonstrated exploratory data analysis (thresholding, normalization, group-by) using `@id` references for fields.
- Visualized a numeric field provided via its `@id`.

This approach ensures reproducible, scalable, and interoperable data exploration workflows for FAIR datasets described by [MLCommons Croissant](https://mlcommons.org/croissant/).

For advanced analysis, further map field `@id` values to their real-world semantics using the schema, e.g. for protein quantities, modifications, and more.